# Minimal LoRA Dual-Head XLM-R with Learned Mixture Gate

Train one LoRA `xlm-roberta-base` backbone with two experts: a 5-class classifier `p(k | x)` and a scalar regressor `s_r(x)`. A learned gate mixes the regressor score with the classifier expectation `s_c(x) = E[y | x]`:

$$g(x) = \sigma(W_g h), \qquad s(x) = g(x)s_r(x) + (1 - g(x))s_c(x).$$

The loss is

$$L = L_{CE} + \lambda L_{Huber}(s_r, y) + \beta L_{Huber}(s, y) + \gamma(s_r - s_c)^2.$$

After training, decode with classifier Bayes-MAE, rounded/tuned `s_r`, or rounded/tuned mixed score `s`.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import json
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Value
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoTokenizer, Trainer, TrainingArguments, XLMRobertaModel, XLMRobertaPreTrainedModel, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_dual_head_mixture_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 1
EVAL_BATCH_SIZE = 1024
# Dual-head LoRA updates ~22M parameters here; keep this conservative.
LR = 2e-5
WARMUP_STEPS = 100
MAX_GRAD_NORM = 1.0
# Keep off for this multi-loss probe unless you have verified it is numerically stable.
FP16 = False

# Loss weights: CE + LAMBDA_HUBER * Huber(s_r, y) + BETA_MIXED_HUBER * Huber(s, y) + GAMMA_CONSISTENCY * agreement.
LAMBDA_HUBER = 0.25
BETA_MIXED_HUBER = 1.00
GAMMA_CONSISTENCY = 0.05
HUBER_DELTA = 0.75

# Optional anti-collapse gate regularizer: ETA_GATE * (mean(g) - GATE_TARGET_USAGE)^2.
# Keep ETA_GATE at 0.0 for the first run; collapse is acceptable if validation improves.
ETA_GATE = 0.0
GATE_TARGET_USAGE = 0.30
GATE_COLLAPSE_LOW = 0.05
GATE_COLLAPSE_HIGH = 0.95

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data and tokenization

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
y_val = val_df["label"].to_numpy(dtype=int)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenize_dual(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["class_labels"] = [int(x) for x in batch["label"]]
    out["score_labels"] = [float(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out


def to_hf_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_dual, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("class_labels", Value("int64"))
    ds = ds.cast_column("score_labels", Value("float32"))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds


train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)

## Model and trainer

In [ ]:
class XLMRDualHeadMixtureModel(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
        hidden = config.hidden_size
        self.num_labels = config.num_labels
        self.lambda_huber = getattr(config, "lambda_huber", LAMBDA_HUBER)
        self.beta_mixed_huber = getattr(config, "beta_mixed_huber", BETA_MIXED_HUBER)
        self.gamma_consistency = getattr(config, "gamma_consistency", GAMMA_CONSISTENCY)
        self.eta_gate = getattr(config, "eta_gate", ETA_GATE)
        self.gate_target_usage = getattr(config, "gate_target_usage", GATE_TARGET_USAGE)
        self.huber_delta = getattr(config, "huber_delta", HUBER_DELTA)
        self.register_buffer("class_values", torch.arange(self.num_labels, dtype=torch.float32), persistent=False)

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, self.num_labels),
        )
        self.regressor = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, 1),
        )
        self.gate = nn.Linear(hidden, 1)
        self.post_init()
        nn.init.normal_(self.regressor[-1].weight, mean=0.0, std=1e-3)
        # raw bias 0 -> 4 * sigmoid(0) = score 2.
        nn.init.constant_(self.regressor[-1].bias, 0.0)
        nn.init.normal_(self.gate.weight, mean=0.0, std=1e-3)
        initial_gate_logit = np.log(GATE_TARGET_USAGE / (1.0 - GATE_TARGET_USAGE))
        nn.init.constant_(self.gate.bias, float(initial_gate_logit))
        self.loss_components = {}

    def forward(self, input_ids=None, attention_mask=None, class_labels=None, score_labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0, :])
        class_logits = self.classifier(pooled)
        # Keep early scalar predictions sane. This avoids huge Huber/consistency losses
        # from randomly initialized scalar heads before they calibrate.
        reg_scores = 4.0 * torch.sigmoid(self.regressor(pooled).squeeze(-1))
        gates = torch.sigmoid(self.gate(pooled)).squeeze(-1)
        probs = F.softmax(class_logits, dim=-1)
        cls_scores = probs @ self.class_values.to(probs.device)
        mixed_scores = gates * reg_scores + (1.0 - gates) * cls_scores

        loss = None
        ce_loss = huber_reg_loss = huber_mixed_loss = consistency_loss = gate_loss = None
        if class_labels is not None and score_labels is not None:
            class_labels = class_labels.long().view(-1)
            score_labels = score_labels.float().view_as(reg_scores)
            ce_loss = F.cross_entropy(class_logits, class_labels)
            huber_reg_loss = F.huber_loss(reg_scores, score_labels, delta=self.huber_delta)
            huber_mixed_loss = F.huber_loss(mixed_scores, score_labels, delta=self.huber_delta)
            consistency_loss = torch.square(reg_scores - cls_scores).mean()
            gate_loss = torch.square(gates.mean() - self.gate_target_usage)
            loss = (
                ce_loss
                + self.lambda_huber * huber_reg_loss
                + self.beta_mixed_huber * huber_mixed_loss
                + self.gamma_consistency * consistency_loss
                + self.eta_gate * gate_loss
            )
            self.loss_components = {
                "ce_loss": ce_loss.detach(),
                "huber_reg_loss": huber_reg_loss.detach(),
                "huber_mixed_loss": huber_mixed_loss.detach(),
                "consistency_loss": consistency_loss.detach(),
                "gate_loss": gate_loss.detach(),
            }

        # Trainer expects one prediction tensor. Columns 0..4 are classifier logits;
        # columns 5..8 are s_r, s_c, s_mix, and gate.
        logits = torch.cat(
            [class_logits, reg_scores.unsqueeze(-1), cls_scores.unsqueeze(-1), mixed_scores.unsqueeze(-1), gates.unsqueeze(-1)],
            dim=-1,
        )
        return {"loss": loss, "logits": logits}


def make_model():
    config = AutoConfig.from_pretrained(MODEL_ID, num_labels=N_CLASSES)
    config.lambda_huber = LAMBDA_HUBER
    config.beta_mixed_huber = BETA_MIXED_HUBER
    config.gamma_consistency = GAMMA_CONSISTENCY
    config.eta_gate = ETA_GATE
    config.gate_target_usage = GATE_TARGET_USAGE
    config.huber_delta = HUBER_DELTA
    model = XLMRDualHeadMixtureModel.from_pretrained(MODEL_ID, config=config)
    lora_config = LoraConfig(
        r=128,
        lora_alpha=64,
        target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        modules_to_save=["classifier", "regressor", "gate"],
        lora_dropout=0.01,
        task_type="SEQ_CLS",
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
    logits = np.clip(logits, -50.0, 50.0)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(cls - classes), axis=1) for cls in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def apply_thresholds(scores, thresholds):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    thresholds = np.asarray(thresholds, dtype=np.float32)
    return np.searchsorted(thresholds, scores, side="right").astype(int)


def tune_mae_thresholds(scores, labels, n_classes=N_CLASSES):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_scores, group_starts = np.unique(sorted_scores, return_index=True)
    group_ends = np.r_[group_starts[1:], len(sorted_scores)]
    n_groups = len(unique_scores)

    group_cost = np.zeros((n_classes, n_groups), dtype=np.float64)
    for g, (start, end) in enumerate(zip(group_starts, group_ends)):
        y = sorted_labels[start:end]
        for cls in range(n_classes):
            group_cost[cls, g] = np.abs(cls - y).sum()

    prefix_cost = np.c_[np.zeros(n_classes), np.cumsum(group_cost, axis=1)]
    dp = np.full((n_classes, n_groups + 1), np.inf, dtype=np.float64)
    back = np.zeros((n_classes, n_groups + 1), dtype=int)
    dp[0] = prefix_cost[0]

    for cls in range(1, n_classes):
        best_value = np.inf
        best_split = 0
        for j in range(n_groups + 1):
            candidate = dp[cls - 1, j] - prefix_cost[cls, j]
            if candidate < best_value:
                best_value = candidate
                best_split = j
            dp[cls, j] = prefix_cost[cls, j] + best_value
            back[cls, j] = best_split

    cuts = []
    j = n_groups
    for cls in range(n_classes - 1, 0, -1):
        j = back[cls, j]
        cuts.append(j)
    cuts = cuts[::-1]

    thresholds = []
    eps = 1e-6
    for cut in cuts:
        if cut <= 0:
            thresholds.append(float(unique_scores[0] - eps))
        elif cut >= n_groups:
            thresholds.append(float(unique_scores[-1] + eps))
        else:
            thresholds.append(float((unique_scores[cut - 1] + unique_scores[cut]) / 2.0))
    return np.array(thresholds, dtype=np.float32), apply_thresholds(scores, thresholds)


def split_predictions(predictions):
    predictions = np.asarray(predictions, dtype=np.float64)
    if not np.isfinite(predictions).all():
        bad = int((~np.isfinite(predictions)).sum())
        print(f"Warning: replacing {bad} non-finite prediction values before metrics.")
        predictions = np.nan_to_num(predictions, nan=0.0, posinf=N_CLASSES - 1, neginf=0.0)
    class_logits = predictions[:, :N_CLASSES]
    reg_scores = np.clip(predictions[:, N_CLASSES], 0.0, N_CLASSES - 1)
    cls_scores = np.clip(predictions[:, N_CLASSES + 1], 0.0, N_CLASSES - 1)
    mixed_scores = np.clip(predictions[:, N_CLASSES + 2], 0.0, N_CLASSES - 1)
    gates = np.clip(predictions[:, N_CLASSES + 3], 0.0, 1.0)
    probs = softmax_np(class_logits)
    # Recompute the classifier expectation from probabilities for numerical consistency.
    expected_scores = probs @ np.arange(N_CLASSES)
    return class_logits, probs, reg_scores, expected_scores, mixed_scores, gates


def summarize_gate(gates, low=GATE_COLLAPSE_LOW, high=GATE_COLLAPSE_HIGH):
    gates = np.asarray(gates, dtype=np.float64).reshape(-1)
    q = np.quantile(gates, [0, 0.1, 0.5, 0.9, 1.0])
    mean_gate = float(gates.mean())
    if mean_gate <= low:
        status = "collapsed_to_classifier"
    elif mean_gate >= high:
        status = "collapsed_to_regressor"
    else:
        status = "mixed"
    return {
        "mean": mean_gate,
        "std": float(gates.std()),
        "q0": float(q[0]),
        "q10": float(q[1]),
        "q50": float(q[2]),
        "q90": float(q[3]),
        "q100": float(q[4]),
        "frac_lt_0.05": float(np.mean(gates < low)),
        "frac_gt_0.95": float(np.mean(gates > high)),
        "status": status,
    }


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    class_labels, score_labels = labels
    y_true = np.asarray(score_labels).reshape(-1)
    _, probs, reg_scores, expected_scores, mixed_scores, gates = split_predictions(predictions)
    map_preds = probs.argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(probs)
    reg_preds = np.rint(np.clip(reg_scores, 0, N_CLASSES - 1)).astype(int)
    mixed_preds = np.rint(np.clip(mixed_scores, 0, N_CLASSES - 1)).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, map_preds)),
        "map_mae": float(mean_absolute_error(y_true, map_preds)),
        "bayes_mae": float(mean_absolute_error(y_true, bayes_preds)),
        "reg_rounded_mae": float(mean_absolute_error(y_true, reg_preds)),
        "mixed_rounded_mae": float(mean_absolute_error(y_true, mixed_preds)),
        "expected_score_mae": float(mean_absolute_error(y_true, expected_scores)),
        "reg_score_mae": float(mean_absolute_error(y_true, reg_scores)),
        "mixed_score_mae": float(mean_absolute_error(y_true, mixed_scores)),
        "mean_gate": float(np.mean(gates)),
        "gate_std": float(np.std(gates)),
        "gate_frac_lt_0.05": float(np.mean(gates < GATE_COLLAPSE_LOW)),
        "gate_frac_gt_0.95": float(np.mean(gates > GATE_COLLAPSE_HIGH)),
        "head_agreement_mse": float(np.mean(np.square(reg_scores - expected_scores))),
    }


def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        max_grad_norm=MAX_GRAD_NORM,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        label_names=["class_labels", "score_labels"],
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Train

In [ ]:
class ComponentLoggingTrainer(Trainer):
    @staticmethod
    def _loss_components(model):
        candidates = [model]
        base_model = getattr(model, "base_model", None)
        if base_model is not None:
            candidates.append(base_model)
            nested = getattr(base_model, "model", None)
            if nested is not None:
                candidates.append(nested)
        for candidate in candidates:
            components = getattr(candidate, "loss_components", None)
            if components:
                return components
        return {}

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs["loss"]
        if self.state.global_step % max(1, self.args.logging_steps) == 0:
            logs = {}
            components = self._loss_components(model)
            for key in ["ce_loss", "huber_reg_loss", "huber_mixed_loss", "consistency_loss", "gate_loss"]:
                value = components.get(key)
                if value is not None:
                    logs[f"train/{key}"] = float(value.detach().cpu())
            if logs:
                self.log(logs)
        return (loss, outputs) if return_outputs else loss


def debug_one_batch(model, dataset):
    batch = dataset.select(range(min(2, len(dataset))))[:]
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items() if hasattr(v, "to")}
    model.train()
    model.zero_grad(set_to_none=True)
    out = model(**batch)
    loss = out["loss"]
    logits = out["logits"]
    print("preflight loss:", float(loss.detach().cpu()))
    print("preflight logits finite:", bool(torch.isfinite(logits).all().detach().cpu()))
    print("preflight prediction ranges:", {
        "reg": [float(logits[:, N_CLASSES].min().detach().cpu()), float(logits[:, N_CLASSES].max().detach().cpu())],
        "cls_score": [float(logits[:, N_CLASSES + 1].min().detach().cpu()), float(logits[:, N_CLASSES + 1].max().detach().cpu())],
        "mixed": [float(logits[:, N_CLASSES + 2].min().detach().cpu()), float(logits[:, N_CLASSES + 2].max().detach().cpu())],
        "gate": [float(logits[:, N_CLASSES + 3].min().detach().cpu()), float(logits[:, N_CLASSES + 3].max().detach().cpu())],
    })
    loss.backward()

    grad_sums = {"classifier": 0.0, "regressor": 0.0, "gate": 0.0, "lora": 0.0}
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        grad = float(param.grad.detach().abs().sum().cpu())
        if "classifier" in name:
            grad_sums["classifier"] += grad
        if "regressor" in name:
            grad_sums["regressor"] += grad
        if "gate" in name:
            grad_sums["gate"] += grad
        if "lora_" in name:
            grad_sums["lora"] += grad
    model.zero_grad(set_to_none=True)
    print("preflight grad sums:", grad_sums)
    return grad_sums


model = make_model()
if torch.cuda.is_available():
    model.to("cuda")
debug_one_batch(model, train_ds)
trainer = ComponentLoggingTrainer(
    model=model,
    args=make_training_args("dual_head_mixture_1epoch"),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()
# metrics = trainer.evaluate()
# metrics

## Validation decoding diagnostics

In [ ]:
val_predictions = trainer.predict(val_ds).predictions
val_class_logits, val_probs, val_reg_scores, val_expected, val_mixed_scores, val_gates = split_predictions(val_predictions)
val_reg_scores_clipped = np.clip(val_reg_scores, 0, N_CLASSES - 1)
val_mixed_scores_clipped = np.clip(val_mixed_scores, 0, N_CLASSES - 1)

map_labels = val_probs.argmax(axis=1).astype(int)
bayes_labels = bayes_mae_decode(val_probs)
reg_round_labels = np.rint(val_reg_scores_clipped).astype(int)
mixed_round_labels = np.rint(val_mixed_scores_clipped).astype(int)
reg_thresholds, reg_tuned_labels = tune_mae_thresholds(val_reg_scores_clipped, y_val)
mixed_thresholds, mixed_tuned_labels = tune_mae_thresholds(val_mixed_scores_clipped, y_val)

summary = pd.DataFrame(
    [
        {"decoder": "classifier_map", "mae": mean_absolute_error(y_val, map_labels), "counts": np.bincount(map_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "classifier_bayes_mae", "mae": mean_absolute_error(y_val, bayes_labels), "counts": np.bincount(bayes_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "regressor_round", "mae": mean_absolute_error(y_val, reg_round_labels), "counts": np.bincount(reg_round_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "regressor_tuned_thresholds", "mae": mean_absolute_error(y_val, reg_tuned_labels), "counts": np.bincount(reg_tuned_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "mixed_score_round", "mae": mean_absolute_error(y_val, mixed_round_labels), "counts": np.bincount(mixed_round_labels, minlength=N_CLASSES).tolist()},
        {"decoder": "mixed_score_tuned_thresholds", "mae": mean_absolute_error(y_val, mixed_tuned_labels), "counts": np.bincount(mixed_tuned_labels, minlength=N_CLASSES).tolist()},
    ]
)
display(summary.sort_values("mae"))

print("reg thresholds:", reg_thresholds.tolist())
print("mixed thresholds:", mixed_thresholds.tolist())
gate_summary = summarize_gate(val_gates)
print("head agreement MSE:", float(np.mean(np.square(val_reg_scores - val_expected))))
print("corr s_r(x), s_c(x):", float(np.corrcoef(val_reg_scores, val_expected)[0, 1]))
print("gate summary:", gate_summary)
if gate_summary["status"] != "mixed":
    print("Gate collapse check:", gate_summary["status"], "- acceptable if validation MAE is best; otherwise try ETA_GATE > 0.")

In [ ]:
probe = pd.DataFrame(
    {
        "y": y_val,
        "reg_score": val_reg_scores,
        "classifier_score": val_expected,
        "mixed_score": val_mixed_scores,
        "gate": val_gates,
        "classifier_bayes": bayes_labels,
        "reg_tuned": reg_tuned_labels,
        "mixed_tuned": mixed_tuned_labels,
        "head_disagreement": val_reg_scores - val_expected,
        "bayes_err": np.abs(bayes_labels - y_val),
        "reg_tuned_err": np.abs(reg_tuned_labels - y_val),
        "mixed_tuned_err": np.abs(mixed_tuned_labels - y_val),
    }
)

display(
    probe.assign(abs_head_disagreement=probe["head_disagreement"].abs())
    .sort_values("abs_head_disagreement", ascending=False)
    .head(20)
)

pd.crosstab(
    pd.Series(bayes_labels, name="classifier_bayes"),
    pd.Series(mixed_tuned_labels, name="mixed_tuned"),
    margins=True,
)

## Save model and consistency config

In [ ]:
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config_path = OUTPUT_DIR / "dual_head_mixture_config.json"
config_path.write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "n_classes": N_CLASSES,
            "lambda_huber": LAMBDA_HUBER,
            "beta_mixed_huber": BETA_MIXED_HUBER,
            "gamma_consistency": GAMMA_CONSISTENCY,
            "eta_gate": ETA_GATE,
            "gate_target_usage": GATE_TARGET_USAGE,
            "gate_collapse_low": GATE_COLLAPSE_LOW,
            "gate_collapse_high": GATE_COLLAPSE_HIGH,
            "huber_delta": HUBER_DELTA,
            "reg_thresholds": reg_thresholds.tolist() if "reg_thresholds" in globals() else None,
            "mixed_thresholds": mixed_thresholds.tolist() if "mixed_thresholds" in globals() else None,
            "gate_summary": gate_summary if "gate_summary" in globals() else None,
            "recommended_decoder": "mixed_score_tuned_thresholds",
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("model:", final_dir)
print("config:", config_path)

## Optional submission

This writes two useful submission candidates: classifier Bayes-MAE decoding and the tuned learned mixture score.

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_predictions = trainer.predict(test_ds).predictions
    _, test_probs, test_reg_scores, test_expected, test_mixed_scores, test_gates = split_predictions(test_predictions)
    test_mixed_scores_clipped = np.clip(test_mixed_scores, 0, N_CLASSES - 1)

    test_bayes = bayes_mae_decode(test_probs)
    test_mixed_tuned = apply_thresholds(test_mixed_scores_clipped, mixed_thresholds if "mixed_thresholds" in globals() else np.arange(0.5, N_CLASSES - 1, 1.0))

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    bayes_path = OUTPUT_DIR / "submission_dual_head_bayes_mae.csv"
    mixed_path = OUTPUT_DIR / "submission_dual_head_mixed_tuned.csv"
    pd.DataFrame({"id": test_df["id"], "label": test_bayes.astype(int)}).to_csv(bayes_path, index=False)
    pd.DataFrame({"id": test_df["id"], "label": test_mixed_tuned.astype(int)}).to_csv(mixed_path, index=False)
    print("bayes counts:", np.bincount(test_bayes, minlength=N_CLASSES).tolist())
    print("mixed tuned counts:", np.bincount(test_mixed_tuned, minlength=N_CLASSES).tolist())
    test_gate_summary = summarize_gate(test_gates)
    print("test gate summary:", test_gate_summary)
    if test_gate_summary["status"] != "mixed":
        print("Gate collapse check:", test_gate_summary["status"], "- acceptable if validation MAE is best.")
    print(bayes_path)
    print(mixed_path)
else:
    print("No test CSV found:", TEST_CSV)